# مدير ملفاتي — Telegram إلى Google Drive
افتح الدفتر في Google Colab واضغط تشغيل في الخلية الوحيدة. في أول استخدام، أضف `TELEGRAM_BOT_TOKEN` و`OWNER_ID` في لوحة **Secrets** واسمح للدفتر بالوصول إليهما. لا تشارك الدفتر أو الأسرار.

In [ ]:
# واجهة التشغيل اليومية — لا تحتاج إلى تعديل هذا الكود.
import os
import sys
import subprocess
from pathlib import Path

ENGINE_COMMIT = "c76f8b9431d0062eca175b3326691b0ea872c800"
REPO_URL = "https://github.com/Hus-GPT/TelegramDriveBot.git"
REPO_DIR = Path("/content/TelegramDriveBot")

print("مرحباً بك في مدير ملفات Google Drive.")
print("أولاً: التحقق من الإعدادات الآمنة...")

try:
    from google.colab import drive, userdata
    bot_token = userdata.get("TELEGRAM_BOT_TOKEN")
    owner_id = userdata.get("OWNER_ID")
except Exception:
    bot_token, owner_id = "", ""

if not bot_token or not owner_id or not str(owner_id).isdigit():
    print("\nيلزم إعداد سرَّين مرة واحدة في لوحة Secrets في Colab:")
    print("1) TELEGRAM_BOT_TOKEN — رمز البوت من BotFather.")
    print("2) OWNER_ID — رقم حسابك في Telegram.")
    print("لا تضع هذه القيم في الدفتر أو في رسائل المحادثة.")
    raise RuntimeError("الإعدادات المطلوبة غير مكتملة.")

os.environ["TELEGRAM_BOT_TOKEN"] = str(bot_token)
os.environ["OWNER_ID"] = str(owner_id)

print("ثانياً: ربط Google Drive...")
drive.mount("/content/drive")

print("ثالثاً: تجهيز نسخة المحرك المعتمدة...")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", ENGINE_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", "--detach", ENGINE_COMMIT], check=True)

print("رابعاً: تجهيز المكتبات...")
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    capture_output=True,
    text=True,
)
if install.returncode:
    print("تعذّر تجهيز المكتبات. أعد تشغيل الخلية، وإذا تكرر الخطأ أرسل رسالة الخطأ الظاهرة.")
    raise RuntimeError("Dependency setup failed.")

print("بدأ تشغيل البوت. اترك هذه الخلية تعمل، ثم افتح البوت في Telegram وأرسل /start.")
print("مهم: عند توقف جلسة Colab يتوقف البوت مؤقتاً؛ أعد تشغيل هذه الخلية لاستئناف العمل.")

process = subprocess.Popen(
    [sys.executable, "-u", "-m", "app.colab_start"],
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

for line in process.stdout:
    # Keep the notebook output useful without echoing secrets or account identifiers.
    safe_line = line.replace(str(bot_token), "[محجوب]")
    safe_line = safe_line.replace(str(owner_id), "[محجوب]")
    if "ERROR" in safe_line or "WARNING" in safe_line or "Traceback" in safe_line:
        print(safe_line.rstrip())

process.wait()
if process.returncode:
    print("توقف المحرك. تحقق من إعدادات Secrets وربط Drive ثم أعد تشغيل الخلية.")
